In [0]:
%pip install lightgbm==4.7.0 xgboost==3.4.1 python-dotenv==1.2.4

In [0]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar `proyecto` sin instalarlo: el código vive en src/
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 06 · Optimización con seguimiento (tracking)

Buscar hiperparámetros es probar muchas configuraciones; sin registro, a la mitad ya no se sabe cuál era la buena. Cada
ensayo queda en **MLflow** con sus parámetros, métricas y etiquetas. Tres cuidados:

1. Se optimizan **solo los finalistas** del torneo: **LightGBM y XGBoost**.
2. El **espacio de búsqueda** lleva conocimiento de cada algoritmo, y el **presupuesto** se fija antes de empezar.
3. La mejor puntuación de una búsqueda es **optimista** (es el máximo de muchos intentos): se **comprueba** en
   prestatarios que la búsqueda nunca vio.

**Reglas escritas ANTES de mirar los resultados:**

- **Parámetros:** se adoptan los optimizados solo si el intervalo del 95 % de su mejora en AUC-PR (optimizado −
  partida), en los prestatarios reservados, queda **entero por encima de 0**. Si no, se quedan los de partida.
- **Modelo final:** el finalista con mayor AUC-PR en los prestatarios reservados, **si** el intervalo de la diferencia
  entre los dos no cruza el 0. Si lo cruza (empate), se elige **LightGBM**, por ser más rápido y simple de operar.

In [0]:
import json
import sys
import warnings
from pathlib import Path

# Si el kernel se reinicia, se pierde el sys.path de la celda 1: se restablece aquí
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd

from proyecto.config import ANONYMOUS_GROUP_COL, PROCESSED_DIR, RANDOM_STATE, REPORTS_DIR, TARGET
from proyecto.data.split import assert_disjoint_members, group_cv, split_by_member
from proyecto.features.engineering import FINAL_FEATURES
from proyecto.models.train import build_pipeline, candidates, encode_target, feature_frame, metrics_from_proba
from proyecto.models.tuning import MODELS, bootstrap_cost_difference, bootstrap_metric_difference, tune
from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment

with warnings.catch_warnings(record=True) as caught:   # mostramos por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    where = configure_mlflow()
for w in caught:
    print("Aviso:", w.message)
experiment = set_experiment("eco-score-optimizacion")
print("MLflow:", where, "| experimento:", experiment)

> **Si ves `local`:** no hay conexión con Databricks (no hay `.env`, no hay red o se agotó la cuota). El notebook
> funciona igual y guarda todo en `mlflow_local.db` (no se versiona). Cuando el grupo configure `.env`, se vuelve a
> ejecutar y queda en Databricks.

## 1. Reservar prestatarios que la búsqueda no verá

Del entrenamiento se separa el 25 % de los **prestatarios** (por `borrower_id`, semilla 7). La búsqueda usa solo el
75 % restante, con 3 pliegues por prestatario. La prueba final sigue cerrada.

`borrower_id` es un número anónimo: mantiene juntos los préstamos de cada prestatario, pero **no entra al modelo**
(se quita de `X_tune` y `X_val`). Los datos vienen de `train_sin_memberkey.parquet`, que sí está en el repositorio.

In [0]:
train = pd.read_parquet(PROCESSED_DIR / "train_sin_memberkey.parquet")
tune_df, val_df = split_by_member(train, test_size=0.25, seed=7, group_col=ANONYMOUS_GROUP_COL)
assert_disjoint_members(tune_df, val_df, group_col=ANONYMOUS_GROUP_COL)
X_tune, y_tune = feature_frame(tune_df).drop(columns=[ANONYMOUS_GROUP_COL]), encode_target(tune_df[TARGET])
X_val, y_val = feature_frame(val_df).drop(columns=[ANONYMOUS_GROUP_COL]), encode_target(val_df[TARGET])
tune_folds = list(group_cv(n_splits=3, seed=RANDOM_STATE).split(X_tune, y_tune, groups=tune_df[ANONYMOUS_GROUP_COL]))
print(f"búsqueda: {len(tune_df):,} préstamos, {tune_df[ANONYMOUS_GROUP_COL].nunique():,} prestatarios ({y_tune.mean():.2%} impago)")
print(f"reservados: {len(val_df):,} préstamos, {val_df[ANONYMOUS_GROUP_COL].nunique():,} prestatarios ({y_val.mean():.2%} impago)")

## 2. La búsqueda

Optuna usa un muestreador bayesiano (TPE): aprende qué zonas del espacio dieron mejor AUC-PR y explora más ahí.

| Modelo | Parámetro | Rango | Motivo |
|---|---|---|---|
| LightGBM | `n_estimators` / `learning_rate` | 150–600 / 0,01–0,2 (log) | Se compensan entre sí: pocos árboles con tasa alta o muchos con tasa baja |
| LightGBM | `num_leaves` | 15–63 | Complejidad de cada árbol; con ~22.000 filas de búsqueda más de 63 sobreajusta |
| LightGBM | `min_child_samples` | 20–300 | Regularización: hojas con al menos esa cantidad de préstamos |
| Ambos | `subsample` / `colsample_bytree` | 0,6–1 / 0,4–1 | Aleatoriedad por filas y columnas, que reduce la varianza |
| Ambos | `reg_lambda` | 0,001–10 (log) | Penalización L2 de los pesos de las hojas |
| XGBoost | `max_depth` | 3–8 | XGBoost crece por profundidad; más de 8 rara vez ayuda en datos tabulares |
| XGBoost | `min_child_weight` | 1–20 (log) | Regularización equivalente a `min_child_samples` |

**Presupuesto (fijado antes):** 30 ensayos para LightGBM y 20 para XGBoost, que es más lento (5 s contra 2 s por
ajuste en el torneo). El estudio se guarda en `data/processed/optuna_06.db`: si la ejecución se corta, se reanuda.

In [0]:
TRIALS = {"lightgbm": 30, "xgboost": 20}
STORAGE = PROCESSED_DIR / "optuna_06.db"
studies = {}
for name, n_trials in TRIALS.items():
    studies[name] = tune(name, X_tune, y_tune, tune_folds, n_trials=n_trials, storage=STORAGE, metric="pr_auc")
    s = studies[name]
    print(f"{name:10s} mejor AUC-PR en la búsqueda = {s.best_value:.4f} (ensayo {s.best_trial.number} de {n_trials})")

In [0]:
# Registrar en MLflow los ensayos ya completos del estudio reanudado.
# `tune()` no abre MLflow cuando el estudio de optuna_06.db ya está completo (faltan 0),
# así que el experimento quedó vacío. Aquí se registran los ensayos EXISTENTES con
# sus datos reales: parámetros, AUC-PR (valor del ensayo) y su desviación entre pliegues.
# El resto del tablero (log loss, costo...) solo se registró en la sesión original y NO
# se puede inventar. Si el modelo ya está en MLflow, no se duplica.
ya = mlflow.search_runs(experiment_names=[experiment], filter_string="tags.etapa = '06_optimizacion'")
registrados = set(ya["tags.modelo"].dropna()) if "tags.modelo" in ya.columns else set()
for name, study in studies.items():
    if name in registrados:
        print(f"{name}: ya estaba registrado en MLflow, no se duplica")
        continue
    with mlflow.start_run(run_name=f"optuna-{name}") as parent:
        mlflow.set_tags({"etapa": "06_optimizacion", "modelo": name, "metrica_optimizada": "pr_auc"})
        for t in study.trials:
            if t.value is None or not t.state.is_finished():
                continue
            metrics = {"pr_auc": t.value}
            if "pr_auc_std" in t.user_attrs:
                metrics["pr_auc_std"] = t.user_attrs["pr_auc_std"]
            log_run(
                f"{name}-trial-{t.number}", dict(t.params), metrics,
                tags={"etapa": "06_optimizacion", "modelo": name, "metrica_optimizada": "pr_auc"},
                nested=True,
            )
        mlflow.log_metric("best_pr_auc", study.best_value)
        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        study.set_user_attr("mlflow_parent_run_id", parent.info.run_id)
    print(f"{name}: {len(study.trials)} ensayos registrados en {experiment}")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, (name, study) in zip(axes, studies.items()):
    values = [t.value for t in study.trials if t.value is not None]
    ax.plot(values, "o", color="#9aa0a6", label="ensayo")
    ax.plot(np.maximum.accumulate(values), "-", color="#4c72b0", label="mejor hasta ahora")
    ax.set_title(name); ax.set_xlabel("ensayo"); ax.legend()
axes[0].set_ylabel("AUC-PR en validación cruzada (mayor es mejor)")
plt.tight_layout(); plt.show()

## 3. ¿Qué parámetros importaron?

La importancia dice dónde vale la pena seguir buscando y dónde no.

In [0]:
imp = pd.DataFrame({name: pd.Series(optuna.importance.get_param_importances(study)) for name, study in studies.items()})
imp.round(3).fillna("")

## 4. La comprobación honesta

Sobre los **prestatarios reservados** se compara cada finalista con sus parámetros de **partida** (los del torneo)
frente a los **optimizados**. Los dos se entrenan con los mismos datos de búsqueda y se evalúan en prestatarios que
ninguno vio. La diferencia de AUC-PR lleva un intervalo del 95 %, remuestreando **prestatarios** completos (500
veces). Como control se reporta también la diferencia de **costo** con su intervalo (`bootstrap_cost_difference`,
la misma prueba del ejemplo del curso; negativo = el optimizado cuesta menos). La decisión sigue la regla de AUC-PR.

In [0]:
defaults = {n: m for n, (m, _) in candidates().items()}
rows, probas = [], {}
for name, study in studies.items():
    _, make_fn, scale = MODELS[name]
    tuned = build_pipeline(make_fn(study.best_params), scale=scale, **FINAL_FEATURES).fit(X_tune, y_tune)
    default = build_pipeline(defaults[name], scale=scale, **FINAL_FEATURES).fit(X_tune, y_tune)
    p_tuned, p_default = tuned.predict_proba(X_val)[:, 1], default.predict_proba(X_val)[:, 1]
    probas[name] = {"partida": p_default, "optimizado": p_tuned}
    m_tuned, m_default = metrics_from_proba(y_val, p_tuned), metrics_from_proba(y_val, p_default)
    diff, low, high = bootstrap_metric_difference(y_val, p_tuned, p_default, val_df[ANONYMOUS_GROUP_COL], n_boot=500)
    cost_diff, cost_low, cost_high = bootstrap_cost_difference(y_val, p_tuned, p_default, val_df[ANONYMOUS_GROUP_COL])
    rows.append({
        "modelo": name,
        "AUC-PR partida": m_default["pr_auc"], "AUC-PR optimizado": m_tuned["pr_auc"],
        "diferencia (opt. − partida)": diff, "IC 95 % inferior": low, "IC 95 % superior": high,
        "log loss partida": m_default["log_loss"], "log loss optimizado": m_tuned["log_loss"],
        "costo partida": m_default["cost"], "costo optimizado": m_tuned["cost"],
        "diferencia de costo": cost_diff, "costo IC 95 % inferior": cost_low, "costo IC 95 % superior": cost_high,
        "% marcado partida": m_default["flagged_rate"], "% marcado optimizado": m_tuned["flagged_rate"],
        "¿se adoptan los optimizados?": "sí" if low > 0 else ("no: no se distingue" if low <= 0 <= high else "no: empeora"),
    })
check = pd.DataFrame(rows).set_index("modelo")
check.round(4).T

**Cómo leer el intervalo:** si todo el intervalo está por encima de 0, el optimizado ordena mejor con 95 % de
confianza. Si cruza el 0, la mejora **no se distingue del ruido** en estos prestatarios, aunque el número haya
subido.

## 5. Elegir el modelo final (regla escrita arriba)

In [0]:
chosen = {name: ("optimizado" if check.loc[name, "IC 95 % inferior"] > 0 else "partida") for name in studies}
p_lgbm, p_xgb = probas["lightgbm"][chosen["lightgbm"]], probas["xgboost"][chosen["xgboost"]]
diff, low, high = bootstrap_metric_difference(y_val, p_xgb, p_lgbm, val_df[ANONYMOUS_GROUP_COL], n_boot=500)
print(f"Parámetros elegidos: {chosen}")
print(f"XGBoost − LightGBM en AUC-PR (reservados): {diff:+.4f}  IC 95 % [{low:+.4f}; {high:+.4f}]")
if low > 0:
    final_name = "xgboost"
elif high < 0:
    final_name = "lightgbm"
else:
    final_name = "lightgbm"   # empate: el más rápido y simple
print("Modelo final:", final_name, "con parámetros de", chosen[final_name])

## 6. Lo que quedó registrado en MLflow

Cada ensayo es una ejecución anidada. Los cinco mejores por AUC-PR:

In [0]:
runs = mlflow.search_runs(
    experiment_names=[experiment],
    filter_string="tags.etapa = '06_optimizacion' and metrics.pr_auc > 0",
    order_by=["metrics.pr_auc DESC"],
    max_results=5,
)
cols = [c for c in ["tags.modelo", "run_id", "metrics.pr_auc", "metrics.pr_auc_std", "metrics.log_loss", "metrics.cost",
                    "metrics.fit_seconds"] if c in runs.columns]
all_runs = mlflow.search_runs(experiment_names=[experiment], filter_string="tags.etapa = '06_optimizacion'")
print(f"ejecuciones registradas en el experimento (ensayos + madres): {len(all_runs)}")
runs[cols].round(4)

In [0]:
_, make_fn, _ = MODELS[final_name]
final_params = studies[final_name].best_params if chosen[final_name] == "optimizado" else {
    k: v for k, v in defaults[final_name].get_params().items()
    if k in {"n_estimators", "learning_rate", "num_leaves", "max_depth"}
}
best = {
    "modelo_final": final_name,
    "parametros_elegidos": chosen[final_name],
    "parametros": final_params,
    "variables": FINAL_FEATURES,
    "busqueda": {name: {"mejor_pr_auc_cv": s.best_value, "mejores_parametros": s.best_params, "ensayos": len(s.trials),
                        "mlflow_parent_run_id": s.user_attrs.get("mlflow_parent_run_id")} for name, s in studies.items()},
    "tracking": where,
    "experimento": experiment,
}
(REPORTS_DIR / "06_mejores_parametros.json").write_text(json.dumps(best, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
check.to_csv(REPORTS_DIR / "06_optimizacion_tabla.csv")
print(json.dumps({k: best[k] for k in ["modelo_final", "parametros_elegidos", "parametros"]}, indent=2, default=float))

## 7. Lectura y decisión

**La búsqueda:** son 30 ensayos de LightGBM y 20 de XGBoost, con 50 ejecuciones de ensayo más 2 madres registradas en MLflow (local).
La mejor AUC-PR en validación cruzada fue 0,6131 (LightGBM, ensayo 24) y 0,6117 (XGBoost, ensayo 17). Esas cifras son
**optimistas**: son el máximo de muchos intentos.

**La comprobación honesta**, en 8.076 prestatarios que la búsqueda no vio:

| Modelo | AUC-PR partida | AUC-PR optimizado | Diferencia | IC 95 % | Regla |
|---|---|---|---|---|---|
| LightGBM | 0,5999 | 0,6080 | **+0,0082** | [+0,0041; +0,0123] | **Se adoptan los optimizados** |
| XGBoost | 0,6060 | 0,6083 | +0,0022 | [−0,0022; +0,0059] | No se distingue: quedan los de partida |

- **Optimizar LightGBM sí sirvió.** Todo el intervalo está por encima de 0. También mejoran el log loss (0,513 →
  0,509) y el costo (0,520 → 0,507, unos 15.300 USD menos por cada 1.000 solicitudes).
- **Control de costo** (`bootstrap_cost_difference`): LightGBM optimizado cuesta −0,013 por solicitud, IC [−0,023;
  −0,003], también mejor con 95 % de confianza. XGBoost: −0,005, IC [−0,015; +0,004], no se distingue.
- **Carga del analista:** con LightGBM optimizado sube un poco el % de solicitudes marcadas para revisión (61,4 % → 63,0 %).
- **En XGBoost la mejora no se distingue del ruido.** Sus parámetros de partida ya eran buenos.
- **Modelo final: LightGBM optimizado.** Contra XGBoost de partida, la diferencia es −0,0020 a favor de LightGBM (IC
  [−0,0065; +0,0024]). Es un empate, y la regla escrita antes elige LightGBM por ser más rápido y simple de operar.
- **Qué parámetros importaron:** en LightGBM, sobre todo `reg_lambda` (0,47), `min_child_samples` (0,14) y
  `colsample_bytree` (0,13). Es decir, la **regularización**, no el tamaño del modelo.

**Una advertencia sobre el espacio de búsqueda:** los mejores parámetros de LightGBM quedaron casi **en el borde**
del rango (`n_estimators` = 598 de 600 y `learning_rate` = 0,011 con mínimo 0,01). Puede que más árboles con una tasa
más baja rindan algo más. No se amplió el rango después de ver el resultado, porque eso gastaría presupuesto que no se
fijó antes. Queda anotado como mejora posible.

**Lo que no se puede concluir:** el AUC-PR del modelo en préstamos nuevos. Eso solo lo dice la prueba (etapa 7), que
sigue cerrada.

Parámetros guardados en `reports/06_mejores_parametros.json`; tabla en `reports/06_optimizacion_tabla.csv`.